<a href="https://colab.research.google.com/github/Astolfobestwaifu/wrhetn4tnrnetm/blob/main/Base_de_Dados_Bruta_(Dataset).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. **Base de Dados Bruta** (Dataset)

Para atender à regra de proibição de edição manual no Excel, utilize o bloco de texto CSV abaixo.

Ele simula dados imobiliários e de infraestrutura com valores nulos, registros corrompidos e inconsistências propositais:   

Crie um arquivo chamado dados_imoveis_bruto.csv com o seguinte conteúdo:

id,bairro,area_m2,quartos,vagas_garagem,preco_aluguel,ano_construcao
101,Jardim Nova Europa,75.0,2,1,2800,2015
102,Ponte Preta,52.5,1,1,2100,2018
103,Jardim Nova Europa,,2,1,2600,2012
104,Cambuí,110.0,3,2,5500,2020
105,Ponte Preta,45.0,1,0,N/A,2010
106,Jardim Nova Europa,-80.0,2,1,3000,2016
107,Cambuí,120.0,3,2,6200,2021
108,Ponte Preta,60.0,2,1,2400,1800
109,Jardim Nova Europa,75.0,2,1,2800,2015
110,Cambuí,95.0,?,2,4800,2019
111,Ponte Preta,58.0,2,,2350,2017
112,Cambuí,250.0,4,3,12000,2022

**2. Passo a Passo de Execução**

Etapa 1: Ambiente e ImportaçãoAbra um novo notebook no Google Colab.   

 *   Suba o arquivo dados_imoveis_bruto.csv para a área de arquivos temporários do Colab ou carregue-o diretamente via script.
*   Importe as bibliotecas essenciais (pandas, numpy).

   






In [15]:
import numpy as np
import pandas as pd

# Importação tratando caracteres especiais como NaN
df_raw = pd.read_csv("dados_imoveis_bruto.csv", na_values=["N/A", "?"])
print("Formato inicial:", df_raw.shape)
df_raw.head()

Formato inicial: (12, 7)


,id,bairro,area_m2,quartos,vagas_garagem,preco_aluguel,ano_construcao
0,101,Jardim Nova Europa,75.0,2.0,1.0,2800.0,2015
1,102,Ponte Preta,52.5,1.0,1.0,2100.0,2018
2,103,Jardim Nova Europa,NaN,2.0,1.0,2600.0,2012
3,104,Cambuí,110.0,3.0,2.0,5500.0,2020
4,105,Ponte Preta,45.0,1.0,0.0,NaN,2010


**Etapa 2: Diagnóstico de Anomalias e Nulos**

Antes de qualquer alteração, mapeie as inconsistências do dataset de forma programática:   
*   Linhas duplicadas (df.duplicated()).
*   Tipos de dados e contagem de nulos (df.info(), df.isnull().sum()).
*   Valores absurdos/impossíveis (ex: área negativa, ano de construção irreal).





In [16]:
# Verificação de nulos e tipos
print("--- Diagnóstico de Valores Nulos ---")
print(df_raw.isnull().sum())

print("\n--- Linhas Duplicadas ---")
print(df_raw.duplicated(subset=["id"]).sum())

print("\n--- Resumo Estatístico Preliminar ---")
print(df_raw.describe())

--- Diagnóstico de Valores Nulos ---
id                0
bairro            0
area_m2           1
quartos           1
vagas_garagem     1
preco_aluguel     1
ano_construcao    0
dtype: int64

--- Linhas Duplicadas ---
0

--- Resumo Estatístico Preliminar ---
               id     area_m2    quartos  vagas_garagem  preco_aluguel  \
count   12.000000   11.000000  11.000000      11.000000      11.000000   
mean   106.500000   78.227273   2.181818       1.363636    4231.818182   
std      3.605551   77.527854   0.873863       0.809040    2930.381607   
min    101.000000  -80.000000   1.000000       0.000000    2100.000000   
25%    103.750000   55.250000   2.000000       1.000000    2500.000000   
50%    106.500000   75.000000   2.000000       1.000000    2800.000000   
75%    109.250000  102.500000   2.500000       2.000000    5150.000000   
max    112.000000  250.000000   4.000000       3.000000   12000.000000   

       ano_construcao  
count       12.000000  
mean      1998.750000  
std

**Etapa 3: Tratamento e Limpeza Algorítmica (EDA)**

Aplicando estritamente a regra de tratar via código:

1.   Remoção de Duplicações: Eliminar registros duplicados com base no identificador.
2.   Correção de Erros de Entrada: Converter valores negativos (como área negativa) para positivos com .abs().
3.   Filtro de Inconsistências: Tratar anos incompatíveis (ex: imóveis anteriores a 1950 no contexto urbano do bairro).
4.   Imputação de Valores Ausentes:

*   Preencher a mediana da área agrupada por número de quartos.
*   Preencher vagas vazias com zero ou moda.
*   Remover linhas onde a variável-alvo (preco_aluguel) está ausente.

In [17]:
df_clean = df_raw.copy()

# 1. Remover IDs duplicados mantendo a primeira ocorrência
df_clean = df_clean.drop_duplicates(subset=["id"], keep="first")

# 2. Corrigir áreas negativas geradas por erro de digitação
df_clean["area_m2"] = df_clean["area_m2"].abs()

# 3. Tratar anos inválidos (substituir anomalias pela mediana dos anos)
mediana_ano = df_clean.loc[df_clean["ano_construcao"] >= 1950, "ano_construcao"].median()
df_clean.loc[df_clean["ano_construcao"] < 1950, "ano_construcao"] = mediana_ano

# 4. Imputar valores ausentes
# Quartos: preencher pela moda
df_clean["quartos"] = df_clean["quartos"].fillna(df_clean["quartos"].mode()[0]).astype(int)

# Vagas de garagem: ausência indica 0 vagas
df_clean["vagas_garagem"] = df_clean["vagas_garagem"].fillna(0).astype(int)

# Área: imputar usando a média da área pelo número de quartos
df_clean["area_m2"] = df_clean.groupby("quartos")["area_m2"].transform(
    lambda x: x.fillna(x.mean())
)

# 5. Descarte de registros sem o valor do aluguel (variável target)
df_clean = df_clean.dropna(subset=["preco_aluguel"])
df_clean["preco_aluguel"] = df_clean["preco_aluguel"].astype(float)

print("--- Base Final Tratada ---")
df_clean.info()
df_clean

--- Base Final Tratada ---
<class 'pandas.core.frame.DataFrame'>
Index: 11 entries, 0 to 11
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              11 non-null     int64  
 1   bairro          11 non-null     object 
 2   area_m2         11 non-null     float64
 3   quartos         11 non-null     int64  
 4   vagas_garagem   11 non-null     int64  
 5   preco_aluguel   11 non-null     float64
 6   ano_construcao  11 non-null     int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 704.0+ bytes


,id,bairro,area_m2,quartos,vagas_garagem,preco_aluguel,ano_construcao
0,101,Jardim Nova Europa,75.000000,2,1,2800.0,2015
1,102,Ponte Preta,52.500000,1,1,2100.0,2018
2,103,Jardim Nova Europa,73.833333,2,1,2600.0,2012
3,104,Cambuí,110.000000,3,2,5500.0,2020
5,106,Jardim Nova Europa,80.000000,2,1,3000.0,2016
6,107,Cambuí,120.000000,3,2,6200.0,2021
7,108,Ponte Preta,60.000000,2,1,2400.0,2017
8,109,Jardim Nova Europa,75.000000,2,1,2800.0,2015
9,110,Cambuí,95.000000,2,2,4800.0,2019
10,111,Ponte Preta,58.000000,2,0,2350.0,2017


**Etapa 4: Análise Exploratória e Métricas**
Gere insights descritivos rápidos para embasar a tomada de decisão do consultor:

In [18]:
# Média de preço por bairro
analise_bairro = (
    df_clean.groupby("bairro")[["preco_aluguel", "area_m2"]]
    .mean()
    .rename(
        columns={
            "preco_aluguel": "aluguel_medio",
            "area_m2": "area_media",
        }
    )
)

print(analise_bairro)

# Exportação do dataset limpo para uso posterior no modelo
df_clean.to_csv("dados_imoveis_tratados.csv", index=False)

                    aluguel_medio  area_media
bairro                                       
Cambuí                7125.000000  143.750000
Jardim Nova Europa    2800.000000   75.958333
Ponte Preta           2283.333333   56.833333


**Etapa 5: Submissão no GitHub**


1.   No menu superior do Google Colab, clique em Arquivo > Salvar uma cópia no GitHub.
2.   Faça login com sua conta, selecione o repositório da disciplina/projeto.
3.   Adicione uma mensagem de commit explicando as correções feitas:"feat: limpeza de dados e EDA inicial - Desafio do Consultor".  
4.   Garanta que o notebook contenha células de texto (Markdown) explicando cada decisão de imputação e remoção.  
